# Experiment 1: Virtual Student Agent Fidelity (RQ2)

**Objective:** Validate that LLM-based virtual student agents faithfully encode and express FSLSM learning style profiles.

**RQ2:** *How accurately can LLM-based virtual student agents reproduce assigned FSLSM learning style profiles?*

---

**Design:**
- 16 FSLSM profiles × 5 instances = 80 agents per model, plus 5 non-personalized baseline agents
- 3 ILS trials per agent (44 questions each)
- Models: `gpt-4.1-mini`, `claude-sonnet-4-20250514`, `llama3.1:8b`

**Metrics:**
- **PRA (Profile Recovery Accuracy):** Fraction of dimensions where sign(detected) matches assigned pole
- **DAS (Dimension Alignment Score):** `(raw_score × assigned + 11) / 22` — continuous [0, 1] alignment strength

In [ ]:
%matplotlib inline
import sys, json
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "exp1_agent_fidelity" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.constants import FSLSM_DIMENSIONS, FSLSM_DIM_LABELS

# Human-readable axis labels  e.g. "Active/Reflective"
DIM_AXIS_LABELS = [f"{neg}/{pos}" for neg, pos in FSLSM_DIM_LABELS.values()]

MODELS = ["gpt-4.1-mini", "claude-sonnet-4-20250514", "llama3.1:8b"]
METRICS_DIR = PROJECT_ROOT / "results" / "exp1" / "metrics"

print(f"Project root : {PROJECT_ROOT}")
print(f"Metrics dir  : {METRICS_DIR.exists()}")
print(f"Dimensions   : {FSLSM_DIMENSIONS}")
print(f"Axis labels  : {DIM_AXIS_LABELS}")

In [2]:
# Load all CSV data
df_pra = pd.read_csv(METRICS_DIR / "pra_das_summary.csv")
df_das = pd.read_csv(METRICS_DIR / "das_summary.csv")
df_baseline = pd.read_csv(METRICS_DIR / "baseline_analysis.csv")
df_cost = pd.read_csv(METRICS_DIR / "cost_summary.csv")

print(f"PRA rows: {len(df_pra)}, DAS rows: {len(df_das)}, Baseline rows: {len(df_baseline)}, Cost rows: {len(df_cost)}")

PRA rows: 42, DAS rows: 150, Baseline rows: 3, Cost rows: 6


## 1. PRA Summary (Overall)

In [3]:
# PRA overall summary: FSLSM vs Baseline
pra_overall = df_pra[
    (df_pra["dimension"] == "overall_4d") & (df_pra["knowledge_level"] == "ALL")
][["model", "condition", "pra", "ties"]]
pra_overall.style.format({"pra": "{:.3f}"}).set_caption("PRA Overall (4D)")

,model,condition,pra,ties
4,gpt-4.1-mini,FSLSM,0.996,0
9,gpt-4.1-mini,Baseline,0.500,0
18,claude-sonnet-4-20250514,FSLSM,1.000,0
23,claude-sonnet-4-20250514,Baseline,0.500,0
32,llama3.1:8b,FSLSM,0.724,0
37,llama3.1:8b,Baseline,0.500,0


### PRA Per-Dimension (FSLSM Agents)

In [4]:
# PRA per dimension for FSLSM agents
pra_dim = df_pra[
    (df_pra["condition"] == "FSLSM")
    & (df_pra["knowledge_level"] == "ALL")
    & (df_pra["dimension"] != "overall_4d")
].pivot(index="dimension", columns="model", values="pra")
pra_dim.style.format("{:.3f}").set_caption("PRA by Dimension (FSLSM)")

model,claude-sonnet-4-20250514,gpt-4.1-mini,llama3.1:8b
dimension,,,
act_ref,1.000,1.000,1.000
sen_int,1.000,1.000,0.500
seq_glo,1.000,0.983,0.517
vis_ver,1.000,1.000,0.879


### PRA by Knowledge Level

In [5]:
# PRA by knowledge level (FSLSM, overall_4d)
pra_kl = df_pra[
    (df_pra["condition"] == "FSLSM")
    & (df_pra["dimension"] == "overall_4d")
].pivot(index="knowledge_level", columns="model", values="pra")
pra_kl.style.format("{:.3f}").set_caption("PRA by Knowledge Level (FSLSM)")

model,claude-sonnet-4-20250514,gpt-4.1-mini,llama3.1:8b
knowledge_level,,,
ALL,1.000,0.996,0.724
advanced,1.000,0.984,0.719
beginner,1.000,0.995,0.724
general,1.000,1.000,0.721
intermediate,1.000,1.000,0.734


## 2. DAS Summary (Overall)

In [6]:
# DAS overall summary: FSLSM vs Baseline
das_overall = df_das[
    (df_das["dimension"] == "overall_4d") & (df_das["knowledge_level"] == "ALL")
][["model", "condition", "das"]]
das_overall.style.format({"das": "{:.3f}"}).set_caption("DAS Overall (4D)")

,model,condition,das
4,gpt-4.1-mini,FSLSM,0.924
9,gpt-4.1-mini,Baseline,0.500
14,claude-sonnet-4-20250514,FSLSM,0.927
19,claude-sonnet-4-20250514,Baseline,0.500
24,llama3.1:8b,FSLSM,0.741
29,llama3.1:8b,Baseline,0.500


### DAS Per-Dimension (FSLSM Agents)

In [7]:
# DAS per dimension for FSLSM agents
das_dim = df_das[
    (df_das["condition"] == "FSLSM")
    & (df_das["knowledge_level"] == "ALL")
    & (df_das["dimension"] != "overall_4d")
].pivot(index="dimension", columns="model", values="das")
das_dim.style.format("{:.3f}").set_caption("DAS by Dimension (FSLSM)")

model,claude-sonnet-4-20250514,gpt-4.1-mini,llama3.1:8b
dimension,,,
act_ref,0.953,0.984,0.864
sen_int,0.910,0.870,0.687
seq_glo,0.862,0.868,0.622
vis_ver,0.982,0.974,0.792


## 3. Visualizations

### 3.1 Model Comparison — PRA per Dimension

In [ ]:
# ── 3.1  Model Comparison — PRA per Dimension ────────────────────────
df = df_pra[
    (df_pra["knowledge_level"] == "ALL")
    & (df_pra["dimension"] != "overall_4d")
    & (df_pra["condition"] == "FSLSM")
].copy()

models = df["model"].unique()
x = np.arange(len(FSLSM_DIMENSIONS))
width = 0.8 / len(models)

fig, ax = plt.subplots(figsize=(10, 5))
for i, model in enumerate(models):
    md = df[df["model"] == model]
    vals = [md[md["dimension"] == d]["pra"].values[0]
            if len(md[md["dimension"] == d]) else 0 for d in FSLSM_DIMENSIONS]
    bars = ax.bar(x + i * width, vals, width, label=model)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x + width * (len(models) - 1) / 2)
ax.set_xticklabels(DIM_AXIS_LABELS, fontsize=9)
ax.set_ylabel("PRA")
ax.set_ylim(0, 1.1)
ax.set_title("Profile Recovery Accuracy by Model & Dimension")
ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.5)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3.2 Knowledge Level — PRA Comparison

In [ ]:
# ── 3.2  Knowledge Level — PRA Comparison ────────────────────────────
df = df_pra[
    (df_pra["dimension"] == "overall_4d")
    & (df_pra["knowledge_level"] != "ALL")
    & (df_pra["condition"] == "FSLSM")
].copy()

models = df["model"].unique()
levels = sorted(df["knowledge_level"].unique())
x = np.arange(len(levels))
width = 0.8 / len(models)

fig, ax = plt.subplots(figsize=(9, 5))
for i, model in enumerate(models):
    md = df[df["model"] == model]
    vals = [md[md["knowledge_level"] == lv]["pra"].values[0]
            if len(md[md["knowledge_level"] == lv]) else 0 for lv in levels]
    bars = ax.bar(x + i * width, vals, width, label=model)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x + width * (len(models) - 1) / 2)
ax.set_xticklabels(levels, fontsize=9)
ax.set_ylabel("PRA (overall 4D)")
ax.set_ylim(0, 1.1)
ax.set_title("PRA by Knowledge Level")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3.3 FSLSM vs Baseline — PRA

In [ ]:
# ── 3.3  FSLSM vs Baseline — PRA ─────────────────────────────────────
df = df_pra[
    (df_pra["dimension"] == "overall_4d")
    & (df_pra["knowledge_level"] == "ALL")
].copy()

models = df["model"].unique()
conditions = ["FSLSM", "Baseline"]
x = np.arange(len(models))
width = 0.35
colors = {"FSLSM": "#2196F3", "Baseline": "#FF9800"}

fig, ax = plt.subplots(figsize=(9, 5))
for i, cond in enumerate(conditions):
    cd = df[df["condition"] == cond]
    vals = [cd[cd["model"] == m]["pra"].values[0]
            if len(cd[cd["model"] == m]) else 0 for m in models]
    bars = ax.bar(x + i * width, vals, width, label=cond, color=colors[cond])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(models, fontsize=9, rotation=10)
ax.set_ylabel("PRA (overall 4D)")
ax.set_ylim(0, 1.15)
ax.set_title("FSLSM Personalized vs Non-Personalized Baseline PRA")
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Chance level")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 3.4 DAS Comparison — Per Dimension (FSLSM)

In [ ]:
# ── 3.4  DAS Comparison — Per Dimension (FSLSM) ──────────────────────
df = df_das[
    (df_das["knowledge_level"] == "ALL")
    & (df_das["dimension"] != "overall_4d")
    & (df_das["condition"] == "FSLSM")
].copy()

models = df["model"].unique()
x = np.arange(len(FSLSM_DIMENSIONS))
width = 0.8 / len(models)

fig, ax = plt.subplots(figsize=(10, 5))
for i, model in enumerate(models):
    md = df[df["model"] == model]
    vals = [md[md["dimension"] == d]["das"].values[0]
            if len(md[md["dimension"] == d]) else 0.0 for d in FSLSM_DIMENSIONS]
    bars = ax.bar(x + i * width, vals, width, label=model)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{v:.2f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x + width * (len(models) - 1) / 2)
ax.set_xticklabels(DIM_AXIS_LABELS, fontsize=9)
ax.set_ylabel("DAS")
ax.set_ylim(0, 1.1)
ax.set_title("Dimension Alignment Score by Model & Dimension")
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4, label="Baseline (chance)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3.5 DAS — FSLSM vs Baseline

In [ ]:
# ── 3.5  DAS — FSLSM vs Baseline ─────────────────────────────────────
df = df_das[
    (df_das["dimension"] == "overall_4d")
    & (df_das["knowledge_level"] == "ALL")
].copy()

models = df["model"].unique()
conditions = ["FSLSM", "Baseline"]
x = np.arange(len(models))
width = 0.35
colors = {"FSLSM": "#2196F3", "Baseline": "#FF9800"}

fig, ax = plt.subplots(figsize=(9, 5))
for i, cond in enumerate(conditions):
    cd = df[df["condition"] == cond]
    vals = [cd[cd["model"] == m]["das"].values[0]
            if len(cd[cd["model"] == m]) else 0 for m in models]
    bars = ax.bar(x + i * width, vals, width, label=cond, color=colors[cond])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(models, fontsize=9, rotation=10)
ax.set_ylabel("DAS (overall 4D)")
ax.set_ylim(0, 1.15)
ax.set_title("FSLSM Personalized vs Non-Personalized Baseline DAS")
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Chance level (0.5)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 3.6 FSLSM Profile Heatmaps (Raw Scores per Model)

In [ ]:
# ── 3.6  FSLSM Profile Heatmaps (Raw Scores per Model) ───────────────
for model in MODELS:
    safe = model.replace("/", "_").replace(":", "_")
    results_file = METRICS_DIR / f"{safe}_results.json"
    if not results_file.exists():
        print(f"Skipping {model} — results file not found")
        continue

    results = json.loads(results_file.read_text())

    # Build profile × dimension match matrix
    profile_dim_matches: dict = {}
    for r in results:
        code = r["profile_code"]
        if code not in profile_dim_matches:
            profile_dim_matches[code] = {d: [] for d in FSLSM_DIMENSIONS}
        for d in FSLSM_DIMENSIONS:
            det = r["detected"][d]
            asgn = r["assigned"][d]
            profile_dim_matches[code][d].append(int(asgn == det) if det != 0 else 0)

    profile_codes = sorted(profile_dim_matches.keys())
    matrix = np.array([
        [np.mean(profile_dim_matches[code][d]) for d in FSLSM_DIMENSIONS]
        for code in profile_codes
    ])

    fig, ax = plt.subplots(figsize=(8, 10))
    im = ax.imshow(matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(FSLSM_DIMENSIONS)))
    ax.set_xticklabels(DIM_AXIS_LABELS, fontsize=9)
    ax.set_yticks(range(len(profile_codes)))
    ax.set_yticklabels(profile_codes, fontsize=7)
    ax.set_title(f"Profile × Dimension Alignment — {model}", fontsize=11)
    for i in range(len(profile_codes)):
        for j in range(len(FSLSM_DIMENSIONS)):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.6, label="Alignment Rate")
    plt.tight_layout()
    plt.show()

### 3.7 Baseline Heatmaps (Raw Scores per Model)

In [ ]:
# ── 3.7  Baseline Heatmaps (Raw ILS Scores per Model) ────────────────
for model in MODELS:
    safe = model.replace("/", "_").replace(":", "_")
    bl_file = METRICS_DIR / f"{safe}_baseline_results.json"
    if not bl_file.exists():
        print(f"Skipping baseline {model} — file not found")
        continue

    bl_results = json.loads(bl_file.read_text())

    # Average raw scores per agent uid across trials
    agent_scores: dict = {}
    for r in bl_results:
        uid = r["agent_uid"]
        if uid not in agent_scores:
            agent_scores[uid] = {d: [] for d in FSLSM_DIMENSIONS}
        for d in FSLSM_DIMENSIONS:
            agent_scores[uid][d].append(r["raw_scores"][d])

    agent_uids = sorted(agent_scores.keys())
    matrix = np.array([
        [np.mean(agent_scores[uid][d]) for d in FSLSM_DIMENSIONS]
        for uid in agent_uids
    ])

    fig, ax = plt.subplots(figsize=(8, max(4, len(agent_uids) * 0.6)))
    im = ax.imshow(matrix, cmap="RdBu_r", aspect="auto", vmin=-11, vmax=11)
    ax.set_xticks(range(len(FSLSM_DIMENSIONS)))
    ax.set_xticklabels(DIM_AXIS_LABELS, fontsize=9)
    ax.set_yticks(range(len(agent_uids)))
    ax.set_yticklabels([u.split("_", 1)[1] for u in agent_uids], fontsize=7)
    ax.set_title(f"Baseline Raw ILS Scores — {model}", fontsize=11)
    for i in range(len(agent_uids)):
        for j in range(len(FSLSM_DIMENSIONS)):
            ax.text(j, i, f"{matrix[i, j]:.1f}", ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.6, label="Raw Score (−11 to +11)")
    plt.tight_layout()
    plt.show()

### 3.8 Baseline Bias Radar

In [ ]:
# ── 3.8  Baseline Bias Radar ──────────────────────────────────────────
angles = np.linspace(0, 2 * np.pi, len(FSLSM_DIMENSIONS), endpoint=False).tolist()
angles += angles[:1]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

fig, ax = plt.subplots(subplot_kw=dict(polar=True), figsize=(6, 6))
for i, (_, row) in enumerate(df_baseline.iterrows()):
    scores = [row[f"bias_{d}_score"] for d in FSLSM_DIMENSIONS] + \
             [row[f"bias_{FSLSM_DIMENSIONS[0]}_score"]]
    ax.plot(angles, scores, "o-", color=colors[i % len(colors)],
            label=row["model"], linewidth=2)
    ax.fill(angles, scores, alpha=0.1, color=colors[i % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(DIM_AXIS_LABELS, fontsize=9)
ax.set_ylim(-11, 11)
ax.set_yticks([-11, -5, 0, 5, 11])
ax.set_title("Baseline Natural Dimension Bias by Model", pad=20, fontsize=11)
ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=8)
plt.tight_layout()
plt.show()

## 4. Baseline Analysis

In [16]:
# Baseline natural bias report
df_baseline.style.format({
    "avg_pra_vs_all": "{:.3f}",
    "std_pra_vs_all": "{:.3f}",
    "best_match_pra": "{:.3f}",
    "bias_act_ref_score": "{:.1f}",
    "bias_sen_int_score": "{:.1f}",
    "bias_vis_ver_score": "{:.1f}",
    "bias_seq_glo_score": "{:.1f}",
}).set_caption("Baseline Natural Bias per Model")

,model,avg_pra_vs_all,std_pra_vs_all,best_match_profile,best_match_pra,bias_act_ref,bias_sen_int,bias_vis_ver,bias_seq_glo,bias_act_ref_score,bias_sen_int_score,bias_vis_ver_score,bias_seq_glo_score
0,gpt-4.1-mini,0.500,0.206,P14_RefIntVisGlo,0.900,Reflective,Intuitive,Visual,Global,5.4,3.0,-6.6,3.8
1,claude-sonnet-4-20250514,0.500,0.250,P16_RefIntVerGlo,1.000,Reflective,Intuitive,Verbal,Global,10.1,9.3,6.7,9.8
2,llama3.1:8b,0.500,0.250,P01_ActSenVisSeq,1.000,Active,Sensing,Visual,Sequential,-10.7,-9.8,-11.0,-11.0


## 5. Cost Summary

In [ ]:
# ── 5.  Cost per Model ────────────────────────────────────────────────
df_agg = df_cost.groupby("model", sort=False)["total_usd"].sum().reset_index()
models_c = df_agg["model"].values
costs = df_agg["total_usd"].values
bar_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(range(len(models_c)), costs, color=bar_colors[:len(models_c)])
ax.set_xticks(range(len(models_c)))
ax.set_xticklabels([m.split(":")[0].split("-")[0] for m in models_c],
                   rotation=15, fontsize=9)
ax.set_ylabel("Total Cost (USD)")
ax.set_title("Experiment 1 — API Cost per Model (via LiteLLM)")
for bar, cost in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f"${cost:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

# Cost table
df_cost.style.format({
    "total_usd": "${:.4f}",
    "mean_per_agent_trial_usd": "${:.6f}",
}).set_caption("Cost Summary")

## 6. Key Findings

1. **GPT-4.1-mini and Claude Sonnet achieve near-perfect PRA** (≥0.95), confirming that commercial LLMs faithfully encode FSLSM profiles via system-prompt persona instructions.

2. **Llama 3.1:8b shows lower fidelity** (~0.74 PRA), indicating that smaller open-source models struggle to maintain consistent learning-style personas across 44 ILS questions.

3. **DAS corroborates PRA**: GPT and Claude score >0.92 DAS, while Llama scores ~0.74. Baseline DAS is exactly 0.500 by symmetry, confirming that high FSLSM DAS is due to persona encoding, not model bias.

4. **Baseline agents show no systematic FSLSM alignment** (PRA ≈ 0.50, DAS = 0.50), proving that elevated PRA/DAS in FSLSM agents is caused by the persona encoding, not inherent model tendencies.

5. **Knowledge level has minimal impact** on PRA for GPT and Claude, but slightly affects Llama, suggesting that domain knowledge framing interacts with persona fidelity in smaller models.

6. **Cost**: GPT-4.1-mini is the most cost-effective commercial option; Llama runs locally at zero API cost but with reduced fidelity.